In [ ]:
import sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = '/content/drive/MyDrive/liya_diploma'
    AI_TOOLKIT = '/content/ai-toolkit'
    sys.path.insert(0, DRIVE_ROOT)
    sys.path.insert(0, AI_TOOLKIT)
    # Colab-only: install runtime deps. Locally these are handled by setup_local.{ps1,sh}.
    get_ipython().system('pip install -q diffusers transformers accelerate requests')
except ModuleNotFoundError:
    # Local run: find project root by looking for scripts/ directory
    _here = Path().resolve()
    DRIVE_ROOT = str(_here if (_here / 'scripts').exists() else _here.parent)
    AI_TOOLKIT = str(Path(DRIVE_ROOT).parent / 'ai-toolkit')
    sys.path.insert(0, DRIVE_ROOT)
    sys.path.insert(0, AI_TOOLKIT)

print(f"DRIVE_ROOT: {DRIVE_ROOT}")

# 50 diverse logo prompts for Experiment 4 commercial comparison
EXP4_PROMPTS = [
    "minimalist coffee shop logo, circular icon, dark green, flat design",
    "tech startup logo, geometric hexagon, blue gradient, bold sans-serif",
    "bakery logo, wheat sheaf icon, warm brown, handcrafted artisan style",
    "fitness brand, lion silhouette, orange and black, bold geometric",
    "law firm logo, balanced scales, navy blue, serif elegant",
    "eco brand, leaf and water droplet, green and teal, clean minimal",
    "photography studio, aperture symbol, monochrome, sleek modern",
    "music label, vinyl record, purple gradient, retro modern",
    "medical clinic, cross and heartbeat, blue and white, professional",
    "restaurant, chef hat and fork, red and gold, classic",
    "real estate, house outline, dark blue, trustworthy minimal",
    "travel agency, compass rose, orange and navy, adventurous",
    "bookstore, open book, burgundy and gold, literary",
    "pet shop, paw print, teal and white, friendly rounded",
    "art gallery, abstract brush stroke, black and coral, creative",
    "yoga studio, lotus flower, lavender and white, serene",
    "gaming company, controller icon, dark purple neon, futuristic",
    "finance app, upward arrow chart, emerald green, trustworthy",
    "delivery service, lightning bolt package, yellow and dark grey, dynamic",
    "hair salon, scissors and comb, rose gold and black, premium",
    "construction company, hard hat, orange and black, bold solid",
    "organic farm, sun over field, green and yellow, natural",
    "coding bootcamp, terminal cursor, dark background cyan, tech",
    "wedding planner, interlinked rings, gold and ivory, elegant",
    "coffee roaster, steam cup bean, dark brown and copper, artisan",
    "surf shop, wave and sun, blue and sandy yellow, coastal",
    "candle brand, flame teardrop, warm amber and cream, cozy",
    "juice bar, orange slice splash, bright orange and green, fresh",
    "security firm, shield with lock, dark grey and blue, strong",
    "toy store, star balloon, bright multicolor, playful",
    "spa resort, lotus in water, soft teal and white, luxury",
    "architecture firm, geometric building outline, charcoal, minimal",
    "flower shop, stylized bloom, pink and green, feminine delicate",
    "wine brand, grape cluster, deep purple and gold, sophisticated",
    "cycling club, wheel spokes, red and white, sporty",
    "accounting firm, balance beam, navy and silver, precise",
    "art supplies, palette with brush, colorful abstract, creative",
    "dog grooming, dog face silhouette, brown and light blue, friendly",
    "language school, speech bubble globe, blue and orange, global",
    "printing shop, ink drop on paper, black and teal, clean",
    "e-commerce, shopping cart pixel, blue and yellow, digital",
    "event planning, balloon ribbon, purple and gold, festive",
    "dental clinic, tooth with sparkle, sky blue and white, clean",
    "vegan restaurant, leaf fork, green and beige, natural",
    "brewery, hops in circle, amber and dark brown, craft",
    "startup incubator, rocket in bulb, blue and green, innovative",
    "cloud storage, cloud with lock, grey and sky blue, tech",
    "insurance company, umbrella shield, dark blue and teal, protective",
    "fashion label, hanger silhouette, black and rose, chic",
    "children school, apple and pencil, red and yellow, friendly",
]
print(f"Loaded {len(EXP4_PROMPTS)} test prompts")

In [ ]:
from diffusers import StableDiffusionXLPipeline
import torch
from pathlib import Path

# UPDATE to the rank with best FID from Experiment 1 (notebook 07)
BEST_SDXL_LORA = f'{DRIVE_ROOT}/results/experiments/sdxl_r16'

out_dir = f'{DRIVE_ROOT}/results/experiments/exp4_sdxl_lora'
Path(out_dir).mkdir(parents=True, exist_ok=True)

pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16,
).to("cuda")
pipe.load_lora_weights(BEST_SDXL_LORA)
pipe.set_progress_bar_config(disable=True)

for i, prompt in enumerate(EXP4_PROMPTS):
    img = pipe(
        f"LOGOIMG {prompt}",
        negative_prompt="photorealistic, blurry, cluttered, complex background",
        generator=torch.Generator().manual_seed(42),
        guidance_scale=7.5,
        num_inference_steps=30,
        height=512, width=512,
    ).images[0]
    img.save(f"{out_dir}/prompt{i:02d}.png")

del pipe; torch.cuda.empty_cache()
print(f"SDXL LoRA: {len(EXP4_PROMPTS)} images \u2192 {out_dir}")

In [ ]:
import requests
from pathlib import Path

RECRAFT_API_KEY = "YOUR_RECRAFT_API_KEY"  # Get free key at recraft.ai

out_dir = f'{DRIVE_ROOT}/results/experiments/exp4_recraft'
Path(out_dir).mkdir(parents=True, exist_ok=True)

for i, prompt in enumerate(EXP4_PROMPTS):
    resp = requests.post(
        "https://external.api.recraft.ai/v1/images/generations",
        headers={
            "Authorization": f"Bearer {RECRAFT_API_KEY}",
            "Content-Type": "application/json",
        },
        json={
            "prompt": f"logo: {prompt}",
            "style": "vector_illustration",
            "width": 512,
            "height": 512,
            "n": 1,
        },
        timeout=60,
    )
    if resp.status_code == 200:
        try:
            img_url = resp.json()["data"][0]["url"]
            img_data = requests.get(img_url, timeout=30).content
            with open(f"{out_dir}/prompt{i:02d}.png", "wb") as f:
                f.write(img_data)
        except Exception as e:
            print(f"Download failed for prompt {i}: {e}")
    else:
        print(f"Recraft error prompt {i}: {resp.status_code} {resp.text[:100]}")

generated = len(list(Path(out_dir).glob("*.png")))
print(f"Recraft: {generated} images → {out_dir}")

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

MODELS = {
    "SD 1.5 Baseline": f'{DRIVE_ROOT}/results/experiments/sd15_baseline',
    "SDXL LoRA (best)": f'{DRIVE_ROOT}/results/experiments/exp4_sdxl_lora',
    "Recraft v3":       f'{DRIVE_ROOT}/results/experiments/exp4_recraft',
}

SHOW = list(range(10))
fig, axes = plt.subplots(len(SHOW), len(MODELS), figsize=(12, 40))

for row, idx in enumerate(SHOW):
    for col, (model_name, img_dir) in enumerate(MODELS.items()):
        suffix = "_v0" if model_name == "SD 1.5 Baseline" else ""
        img_path = f"{img_dir}/prompt{idx:02d}{suffix}.png"
        if Path(img_path).exists():
            axes[row, col].imshow(Image.open(img_path))
        else:
            axes[row, col].text(0.5, 0.5, "Missing", ha='center', va='center')
        axes[row, col].axis('off')
        if row == 0:
            axes[row, col].set_title(model_name, fontsize=10, fontweight='bold')
        if col == 0:
            axes[row, col].set_ylabel(
                EXP4_PROMPTS[idx][:35], fontsize=6, rotation=0,
                labelpad=80, va='center'
            )

plt.suptitle("Experiment 4: Model Comparison (10 prompts × 3 models)", fontsize=13)
plt.tight_layout()
plt.savefig(f'{DRIVE_ROOT}/results/experiments/exp4_comparison_grid.png', dpi=150)
plt.show()